# Run Your Own Container on DesignSafe

Code that runs on your laptop does not automatically run on Stampede3. The cluster offers a fixed set of software modules, and anything outside them has to be packaged before it can run there. A container is that package, and this notebook runs one on a compute node.

Two dapi calls register a **container app** under your account. The app treats the image and the command as job parameters, so this one app runs every container you ever build. The image here is a site-response analysis published from GitHub (`ghcr.io/kks32/python-container-s3`, built by GitHub Actions from [this repository](https://github.com/kks32/python-container-s3)); swap in your own image the same way. [Custom Containers](https://designsafe-ci.github.io/dapi/containers) covers building and publishing images.

In [ ]:
%pip install --quiet --upgrade dapi

**Restart the kernel once after the install**, then run from the next cell.

In [1]:
from pathlib import Path

from dapi import DSClient

ds = DSClient()

# DesignSafe JupyterHub mounts your MyData at ~/MyData; scratch goes there
# since community folders are read-only. Anywhere else, write beside the
# notebook.
mydata = Path.home() / "MyData"
work_root = mydata if mydata.is_dir() else Path.cwd()

/Users/krishna/dev/DesignSafe/Dapi-Tapis/dapi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authentication successful.


TMS credentials ready: frontera, stampede3


## Register the container app, once

`new()` writes the app's two files from dapi's `container` template, and `deploy()` uploads the wrapper to your MyData and registers the app under your account. Rerunning `deploy()` updates the app in place.

In [2]:
import shutil

shutil.rmtree(work_root / "my-container", ignore_errors=True)  # rerun-safe
ds.apps.new("my-container", target_dir=str(work_root), template="container")
result = ds.apps.deploy(str(work_root / "my-container"))
result

Created app 'my-container' from template 'container' at /Users/krishna/MyData/my-container


Updated existing app my-container v0.1.0


{'app_id': 'my-container',
 'version': '0.1.0',
 'container_image': 'tapis://designsafe.storage.default/kks32/apps/my-container/0.1.0/my-container.zip'}

The wrapper the app runs accepts any of three image forms, a `docker://` registry reference pulled on the compute node, a staged `.sif` file, or a staged `docker save` tarball.

In [3]:
app = ds.tapis.apps.getAppLatestVersion(appId="my-container")
print(app.id, app.version, "| owner:", app.owner)

my-container 0.1.0 | owner: kks32


## Stage the job inputs

The container reads and writes `/data`, which is this input directory bind-mounted into it. Everything the analysis writes there is archived when the job ends. This analysis needs no input files, so one placeholder travels along.

In [4]:
work_dir = work_root / "container-inputs"
work_dir.mkdir(parents=True, exist_ok=True)
(work_dir / "README.txt").write_text("input directory for the container job\n")
inputs_uri = (
    "tapis://designsafe.storage.default/"
    + ds.tapis.username
    + "/dapi-demo/container-inputs"
)
ds.files.upload(str(work_dir / "README.txt"), inputs_uri + "/README.txt")
print("inputs at", inputs_uri)

inputs at tapis://designsafe.storage.default/kks32/dapi-demo/container-inputs


## Submit the job

The image and the command are ordinary job parameters. The analysis computes the amplification spectrum of a damped soil layer and checks that the resonant peak lands at the fundamental site frequency, `f0 = Vs / 4H`.

In [5]:
allocation = "DS-Portal-SPARC2026"  # <-- replace with your allocation

job = {
    "name": "container-site-response",
    "appId": "my-container",
    "appVersion": result["version"],
    "execSystemLogicalQueue": "skx-dev",
    "nodeCount": 1,
    "coresPerNode": 1,
    "maxMinutes": 15,
    "fileInputs": [{"name": "Input Directory", "sourceUrl": inputs_uri}],
    "parameterSet": {
        "envVariables": [
            {
                "key": "CONTAINER_IMAGE",
                "value": "docker://ghcr.io/kks32/python-container-s3:latest",
            },
            {"key": "COMMAND", "value": "python3 /opt/app/site_response.py"},
        ],
        "schedulerOptions": [{"name": "TACC Allocation", "arg": "-A " + allocation}],
    },
}
submitted = ds.jobs.submit(job)
final = submitted.monitor(interval=15)
print("final status:", final)

Job submitted successfully. UUID: 39dc6364-76dd-4665-ba4d-2af24704ca81-007



Monitoring Job: 39dc6364-76dd-4665-ba4d-2af24704ca81-007


Waiting for job to start: 0 checks [00:00, ? checks/s]

Waiting for job to start: 0 checks [00:00, ? checks/s, Status: PENDING]

Waiting for job to start: 1 checks [00:15, 15.12s/ checks, Status: PENDING]

...

Monitoring job (Status: ARCHIVING):  10%|█         | 6/60 [01:15<12:31, 13.91s/ checks]

Monitoring job (Status: ARCHIVING):  10%|█         | 6/60 [01:30<12:31, 13.91s/ checks]

Monitoring job (Status: ARCHIVING): 100%|██████████| 60/60 [01:30<00:00, 13.91s/ checks]

Monitoring job (Status: ARCHIVING): 100%|██████████| 60/60 [01:30<00:00,  1.51s/ checks]

	Status: FINISHED
final status: FINISHED


## Read the results

The compute node pulled the image from the registry, ran the command at `/data`, and Tapis archived what it wrote.

In [6]:
import json

report = json.loads(submitted.get_output_content("inputDirectory/site_response.json"))
print(json.dumps(report, indent=2))
assert report["peak_matches_theory"]
print(
    f"\nresonance at {report['f_peak_hz']:.2f} Hz matches f0 = Vs/4H = {report['f0_theory_hz']:.2f} Hz"
)

{
  "profile": {
    "Vs": 200.0,
    "H": 25.0,
    "damping": 0.05
  },
  "f0_theory_hz": 2.0,
  "f_peak_hz": 1.9956114028507126,
  "peak_amplification": 12.748071262358605,
  "peak_matches_theory": true
}

resonance at 2.00 Hz matches f0 = Vs/4H = 2.00 Hz


## Iterate and share

Rebuild and push the image, and the next job picks it up; the app never changes. Pin a tag or digest instead of `latest` when a study must reproduce exactly. Share the app and collaborators run their own images through it with their own allocations, and remove it when you are done experimenting.

```python
ds.tapis.apps.shareApp(appId="my-container", users=["collaborator1"])
ds.tapis.apps.deleteApp(appId="my-container")
```

A containerized job is an ordinary job, so it also drops into a [workflow](https://designsafe-ci.github.io/dapi/workflows) unchanged, one graph node per container.